# 01 · Recommendation Systems: Collaborative, Content-Based & Hybrid

**Phase 6 — Section 6.5 Recommendation Systems | Notebook 1 of 1 (final section)**

### Why this matters
Recommendation systems are one of the most commercially important applications of
everything in this phase: clustering, dimensionality reduction (SVD!), and similarity
metrics (Module 1.1) all converge here. This notebook builds three genuinely different
recommendation strategies on real movie ratings.

### The real dataset: MovieLens (ml-latest-small)

100,836 real ratings from 610 real users across 9,724 movies, plus genre metadata for
content-based filtering.

```
https://raw.githubusercontent.com/sankalpjain99/Movie-recommendation-system/master/ratings.csv
https://raw.githubusercontent.com/sankalpjain99/Movie-recommendation-system/master/movies.csv
```

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ratings = pd.read_csv("https://raw.githubusercontent.com/sankalpjain99/Movie-recommendation-system/master/ratings.csv")
movies = pd.read_csv("https://raw.githubusercontent.com/sankalpjain99/Movie-recommendation-system/master/movies.csv")
print(ratings.shape, movies.shape)
ratings.head()

(100836, 4) (9742, 3)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## 0. The sparsity problem

In [2]:
n_users, n_movies = ratings["userId"].nunique(), ratings["movieId"].nunique()
n_possible = n_users * n_movies
n_actual = len(ratings)
print(f"{n_users} users x {n_movies} movies = {n_possible:,} possible ratings")
print(f"we actually have {n_actual:,} ratings -- {n_actual/n_possible:.2%} filled in")
print("\nThis extreme sparsity is THE central challenge every method below has to work around.")

610 users x 9724 movies = 5,931,640 possible ratings
we actually have 100,836 ratings -- 1.70% filled in

This extreme sparsity is THE central challenge every method below has to work around.


## 1. Building the user-item ratings matrix

In [3]:
# Restrict to movies with at least 20 ratings -- both for meaningful similarity signal
# and to keep the similarity matrices a manageable size (a common, realistic practical step)
movie_counts = ratings["movieId"].value_counts()
popular_movies = movie_counts[movie_counts >= 20].index
ratings_filtered = ratings[ratings["movieId"].isin(popular_movies)]

user_item = ratings_filtered.pivot_table(index="userId", columns="movieId", values="rating")
print(f"filtered matrix: {user_item.shape[0]} users x {user_item.shape[1]} movies")

user_item_filled = user_item.fillna(0)

filtered matrix: 610 users x 1297 movies


## 2. Collaborative Filtering — User-Based

**User-based CF**: find users with SIMILAR taste to the target user (via cosine
similarity, Module 1.1), then recommend what those similar users liked that the target
user hasn't seen yet.

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_item_filled.values)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item.index, columns=user_item.index)

def recommend_user_based(user_id, n=5):
    similar_users = user_similarity_df[user_id].drop(user_id).sort_values(ascending=False)
    top_similar = similar_users.head(20)   # the 20 most similar users

    # weighted average of similar users' ratings, weighted by similarity
    weighted_ratings = user_item.loc[top_similar.index].T.dot(top_similar) / top_similar.sum()

    already_rated = user_item.loc[user_id].dropna().index
    recommendations = weighted_ratings.drop(already_rated, errors="ignore").sort_values(ascending=False)
    return recommendations.head(n)

target_user = 1
recs = recommend_user_based(target_user)
movies.set_index("movieId").loc[recs.index][["title", "genres"]].assign(predicted_rating=recs.values)

,title,genres,predicted_rating
movieId,,,
2,Jumanji (1995),Adventure|Children|Fantasy,NaN
5,Father of the Bride Part II (1995),Comedy,NaN
7,Sabrina (1995),Comedy|Romance,NaN
10,GoldenEye (1995),Action|Adventure|Thriller,NaN
11,"American President, The (1995)",Comedy|Drama|Romance,NaN


## 3. Collaborative Filtering — Item-Based

**Item-based CF**: instead of comparing users, compare MOVIES to each other (do the same
users tend to rate them similarly?), then recommend movies similar to ones the target user
already rated highly. Item-based CF is often more stable in practice, since item
similarity tends to change more slowly over time than an individual user's taste.

In [5]:
item_similarity = cosine_similarity(user_item_filled.T.values)
item_similarity_df = pd.DataFrame(item_similarity, index=user_item.columns, columns=user_item.columns)

def recommend_item_based(user_id, n=5):
    user_ratings = user_item.loc[user_id].dropna()
    scores = pd.Series(0.0, index=user_item.columns)
    weight_sums = pd.Series(0.0, index=user_item.columns)

    for movie_id, rating in user_ratings.items():
        similar_scores = item_similarity_df[movie_id]
        scores += similar_scores * rating
        weight_sums += similar_scores.abs()

    predicted = (scores / weight_sums.replace(0, np.nan)).drop(user_ratings.index, errors="ignore")
    return predicted.sort_values(ascending=False).head(n)

recs_item = recommend_item_based(target_user)
movies.set_index("movieId").loc[recs_item.index][["title", "genres"]].assign(predicted_rating=recs_item.values)

,title,genres,predicted_rating
movieId,,,
96821,"Perks of Being a Wallflower, The (2012)",Drama|Romance,4.503322
1125,"Return of the Pink Panther, The (1975)",Comedy|Crime,4.478689
91529,"Dark Knight Rises, The (2012)",Action|Adventure|Crime|IMAX,4.477007
152081,Zootopia (2016),Action|Adventure|Animation|Children|Comedy,4.476274
112552,Whiplash (2014),Drama,4.475907


## 4. Matrix Factorization — the SVD you derived in Module 1.1, applied for real

Both methods above compute FULL similarity matrices, which gets expensive at scale. Matrix
factorization instead compresses the ratings matrix into a small number of **latent
factors** (recall Module 1.1's SVD notebook!) — hidden dimensions like "prefers action
movies" or "prefers older films" that the model discovers on its own.

In [6]:
from sklearn.decomposition import TruncatedSVD

# center each user's ratings around their own average (removes "some users just rate everything higher")
user_means = user_item.mean(axis=1)
ratings_centered = user_item.sub(user_means, axis=0).fillna(0)

svd = TruncatedSVD(n_components=20, random_state=42)
user_factors = svd.fit_transform(ratings_centered)
item_factors = svd.components_

print(f"compressed {user_item.shape} down to {user_factors.shape} user factors and {item_factors.shape} item factors")
print(f"variance explained by 20 latent factors: {svd.explained_variance_ratio_.sum():.1%}")

reconstructed = user_factors @ item_factors + user_means.values.reshape(-1, 1)
reconstructed_df = pd.DataFrame(reconstructed, index=user_item.index, columns=user_item.columns)

def recommend_svd(user_id, n=5):
    already_rated = user_item.loc[user_id].dropna().index
    predicted = reconstructed_df.loc[user_id].drop(already_rated, errors="ignore")
    return predicted.sort_values(ascending=False).head(n)

recs_svd = recommend_svd(target_user)
movies.set_index("movieId").loc[recs_svd.index][["title", "genres"]].assign(predicted_rating=recs_svd.values.round(2))

compressed (610, 1297) down to (610, 20) user factors and (20, 1297) item factors
variance explained by 20 latent factors: 27.9%


,title,genres,predicted_rating
movieId,,,
778,Trainspotting (1996),Comedy|Crime|Drama,4.70
2683,Austin Powers: The Spy Who Shagged Me (1999),Action|Adventure|Comedy,4.67
344,Ace Ventura: Pet Detective (1994),Comedy,4.64
1653,Gattaca (1997),Drama|Sci-Fi|Thriller,4.64
7147,Big Fish (2003),Drama|Fantasy|Romance,4.62


> 💡 **This is exactly the "SVD recommends the missing ratings" idea previewed in Module
> 1.1's Linear Algebra applications notebook** — now applied to real data with 100x more
> users and movies, with mean-centering added to handle each user's personal rating scale.

## 5. Content-Based Filtering — using genres instead of other users

Every method so far needs OTHER users' ratings. **Content-based filtering** instead uses
the ITEM's own attributes (genres, here) — useful especially for brand-new items with no
ratings yet (the "cold start" problem collaborative methods struggle with).

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

genre_vectorizer = CountVectorizer(tokenizer=lambda x: x.split("|"))
genre_matrix = genre_vectorizer.fit_transform(movies["genres"])
genre_similarity = cosine_similarity(genre_matrix)
genre_similarity_df = pd.DataFrame(genre_similarity, index=movies["movieId"], columns=movies["movieId"])

def recommend_content_based(movie_title, n=5):
    movie_id = movies.loc[movies["title"] == movie_title, "movieId"].values[0]
    similar = genre_similarity_df[movie_id].drop(movie_id).sort_values(ascending=False)
    return movies.set_index("movieId").loc[similar.head(n).index][["title", "genres"]]

recommend_content_based("Toy Story (1995)")

C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,title,genres
movieId,,
103755,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy
65577,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy
4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
91355,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy


## 6. Hybrid — blending collaborative and content-based scores

A hybrid system combines BOTH signals — collaborative filtering's "people like you enjoyed
this" with content-based filtering's "this is similar to what you already liked" — often
more robust than either alone, and able to handle new items via the content-based half.

In [8]:
def recommend_hybrid(user_id, n=5, collab_weight=0.7):
    collab_scores = reconstructed_df.loc[user_id]

    # content-based score: average genre-similarity to the movies this user rated highly (>=4)
    liked_movies = user_item.loc[user_id][user_item.loc[user_id] >= 4].index
    liked_movies = [m for m in liked_movies if m in genre_similarity_df.index]
    if liked_movies:
        content_scores = genre_similarity_df[liked_movies].mean(axis=1)
    else:
        content_scores = pd.Series(0.0, index=user_item.columns)

    # normalize both to [0, 1] before blending, so one doesn't dominate purely from scale
    collab_norm = (collab_scores - collab_scores.min()) / (collab_scores.max() - collab_scores.min())
    content_norm = (content_scores - content_scores.min()) / (content_scores.max() - content_scores.min() + 1e-9)

    combined = collab_weight * collab_norm + (1 - collab_weight) * content_norm.reindex(collab_norm.index).fillna(0)
    already_rated = user_item.loc[user_id].dropna().index
    return combined.drop(already_rated, errors="ignore").sort_values(ascending=False).head(n)

recs_hybrid = recommend_hybrid(target_user)
movies.set_index("movieId").loc[recs_hybrid.index][["title", "genres"]]

,title,genres
movieId,,
2683,Austin Powers: The Spy Who Shagged Me (1999),Action|Adventure|Comedy
778,Trainspotting (1996),Comedy|Crime|Drama
7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy
1262,"Great Escape, The (1963)",Action|Adventure|Drama|War
6503,Charlie's Angels: Full Throttle (2003),Action|Adventure|Comedy|Crime|Thriller


## 🧪 Practice Exercises

**1.** Try `recommend_user_based` and `recommend_item_based` for a different `user_id`
(e.g. `42`). Do the two methods agree on any of the same movies?

**2.** Increase `TruncatedSVD`'s `n_components` from 20 to 50. Does explained variance
increase substantially? At what point do you think adding more components stops helping?

**3.** Change `recommend_hybrid`'s `collab_weight` from 0.7 to 0.3 (favoring content-based
more). How much do the recommendations change for `target_user`?

In [ ]:
# --- Exercise 1 ---
other_user = 42
print("user-based recs:", movies.set_index('movieId').loc[recommend_user_based(other_user).index]['title'].tolist())
print("item-based recs:", movies.set_index('movieId').loc[recommend_item_based(other_user).index]['title'].tolist())

In [ ]:
# --- Exercise 2 ---
svd_50 = TruncatedSVD(n_components=50, random_state=42).fit(ratings_centered)
print(f"20 components: {svd.explained_variance_ratio_.sum():.1%} variance explained")
print(f"50 components: {svd_50.explained_variance_ratio_.sum():.1%} variance explained")

In [ ]:
# --- Exercise 3 ---
recs_content_heavy = recommend_hybrid(target_user, collab_weight=0.3)
print("collab_weight=0.7:", movies.set_index('movieId').loc[recs_hybrid.index]['title'].tolist())
print("collab_weight=0.3:", movies.set_index('movieId').loc[recs_content_heavy.index]['title'].tolist())

---
### ✅ Checkpoint — Section 6.5 & Phase 6 complete!
You should now be able to:
- Explain user-based vs. item-based collaborative filtering
- Apply SVD-based matrix factorization to a real sparse ratings matrix
- Build a content-based recommender using item attributes instead of other users' ratings
- Blend collaborative and content-based scores into a hybrid recommender

## 🎓 Phase 6 complete!
Every technique in this phase found structure in data with NO labels — clusters, compressed
representations, anomalies, item associations, and recommendations — all from real,
GitHub-hosted data: mall customers, wine chemistry, half a million transactions, grocery
baskets, and movie ratings.

**Next phase: Phase 7 — Feature Engineering** — going deeper on the feature-crafting
skills this phase (and Phase 5) already put to heavy use.